# 3 · From the table to answers: `_joint`, `posterior`, `log_likelihood`, `_predict_proba`

**Trusts:** `_unnormalized` (notebook 2).

Each of these is a small step on top of the unnormalized table:

| method | is |
|---|---|
| `_joint(row)` | the table, normalized: P(hidden \| observed) |
| `posterior(data, node)` | one node's marginal of that, for every row |
| `log_likelihood(data)` | Σ over rows of log(table.sum()) |
| `_predict_proba(data, target)` | `posterior` as a DataFrame; masking happens in the base class first |

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools

import numpy as np
import pandas as pd
from toy import (
    CARDS,
    CO,
    DATA,
    EVIDENCE,
    NO2,
    S1,
    S4,
    STRUCTURE,
    TABLES,
    H,
    T,
    cpts,
    network,
    same,
    toy_bn,
)

from cse250a.modules.core import CPT

net = network()
bn = toy_bn()   # DiscreteBN with the hand-picked CPTs, not fitted

## `_joint`: normalize

Row 4 is the prediction situation: CO and NO2 hidden, sensors seen. Dividing the table by its sum turns P(hidden, observed) into **P(hidden | observed)** (Bayes' rule, with the sum as the denominator).

In [ ]:
row = DATA.iloc[4]
hidden, table = bn._unnormalized(row)

joint = table / table.sum()
print("hidden axes:", [str(h) for h in hidden])
print(pd.DataFrame(joint, index=["CO=0", "CO=1"], columns=["NO2=0", "NO2=1"]).round(4))
print("sums to", joint.sum())

same("ours = bn._joint", joint, bn._joint(row)[1])

## `posterior`: one node's marginal

**By hand first.** P(CO = c | row) is row *c* of that table summed across NO2:

In [ ]:
p_co = np.array([joint[0, 0] + joint[0, 1],
                 joint[1, 0] + joint[1, 1]])
print("P(CO | row 4) =", p_co.round(4))

**In general**, the node's axis is `hidden.index(node)`, and we sum over every other axis. With 3 hidden nodes (row 7) that's two axes to sum away.

In [ ]:
def marginal(row, node):
    hidden, joint = bn._joint(row)
    j = hidden.index(node)
    others = tuple(a for a in range(joint.ndim) if a != j)
    return joint.sum(axis=others)

same("general = by hand (row 4, CO)", marginal(DATA.iloc[4], CO), p_co)
print("row 7, three hidden → P(CO):", marginal(DATA.iloc[7], CO).round(4))

**If the node is observed**, the answer is certain: a one-hot on its observed bin. Conditioning on CO = 1 leaves no doubt about CO.

Put both cases in a loop over rows, and that's `posterior`:

In [ ]:
def posterior(data, node):
    out = np.zeros((len(data), CARDS[node]))
    for i, (_, row) in enumerate(data.iterrows()):
        if pd.isna(row[node]):
            out[i] = marginal(row, node)
        else:
            out[i, int(row[node])] = 1
    return out

for node in net.nodes:
    same(f"posterior({node})", posterior(DATA, node), bn.posterior(DATA, node))

## Why it has to be joint: explaining away

S4 responds to both gases. Watch what knowing NO2 does to our belief about CO, with the same sensor readings (hour=1, T=0, S1 missing, S4=1):

In [ ]:
base = pd.Series({H: 1.0, T: 0.0, CO: np.nan, NO2: np.nan, S1: np.nan, S4: 1.0})
def knowing_no2(value):
    r = base.copy()
    r[NO2] = value
    return r

cases = {
    "S4 high, NO2 unknown": base,
    "S4 high, NO2 known low": knowing_no2(0.0),
    "S4 high, NO2 known high": knowing_no2(1.0),
}
for name, r in cases.items():
    print(f"{name:26}  P(CO high) = {marginal(r, CO)[1]:.3f}")

If NO2 is known to be high, it already explains the high S4, so CO becomes *less* likely. If NO2 is known to be low, something else must explain S4, so CO becomes *more* likely. A model that predicted each gas separately couldn't do this. That's why `_unnormalized` builds one table over **all** hidden nodes.

## `log_likelihood`

`table.sum()` = P(row's observed cells | inputs) (notebook 2). The data's log-likelihood is the sum of the logs. Rows are independent given the CPTs.

In [ ]:
per_row = [np.log(bn._unnormalized(row)[1].sum()) for _, row in DATA.iterrows()]
print("per row:", np.round(per_row, 3))
same("Σ log P(observed)", sum(per_row), bn.log_likelihood(DATA))

Rows with more missing cells have a *higher* (less negative) log-likelihood: there's less to explain. So only compare log-likelihoods across models **on the same rows**.

## `_predict_proba` and the query

`bn.predict_proba(data, target, evidence)` lives in the base class `Model`:

1. **mask:** every column outside `evidence` becomes NaN (after checking that no reference is in the evidence and every input is)
2. **`_predict_proba(masked, target)`:** `posterior` on the masked data, as a DataFrame

So a prediction is a posterior where **the references are forced hidden**, even on rows where we have them.

In [ ]:
masked = bn.mask(DATA, CO, EVIDENCE)
display(masked)
same("predict_proba = posterior on masked data",
     bn.predict_proba(DATA, CO, EVIDENCE).to_numpy(), posterior(masked, CO))

The query decides target *and* evidence, so one model answers different questions. Here: drop S4 from the evidence, or predict a sensor instead of a gas.

**Reading the result:** `predict_proba` returns a DataFrame with **one row per row of the data** and **one column per bin of the target**, labeled `0 … k-1`. Each row sums to 1. So `[1]` below selects the column *named* 1, which is P(target = 1 | evidence), i.e. "high" for these binary variables, giving one number per row. `[0]` would be P(low) = 1 − P(high). With 3 bins (low/med/high), "high" would be `[2]`.

In [ ]:
proba = bn.predict_proba(DATA, CO, EVIDENCE)
display(proba.round(3))   # columns = CO's bins; [1] picks the "high" column

In [ ]:
print("P(CO high), full evidence: ", bn.predict_proba(DATA, CO, EVIDENCE)[1].round(3).tolist())
print("P(CO high), without S4:    ", bn.predict_proba(DATA, CO, (H, T, S1))[1].round(3).tolist())
print("P(S4 high | hour, T, S1):  ", bn.predict_proba(DATA, S4, (H, T, S1))[1].round(3).tolist())

And the two guards. A reference in the evidence would let the model read its own answer. An input missing from the evidence can't be summed out, because inputs have no CPT.

In [ ]:
for evidence in [(*EVIDENCE, NO2), (H, S1, S4)]:
    try:
        bn.predict_proba(DATA, CO, evidence)
    except ValueError as e:
        print("ValueError:", e)